# Appendix A: Lie Groups and Robot Kinematics

Source orientation: printed pp. 403-434; PDF pp. 421-452. Chapter question: **What Lie-group facts explain the coordinate rules used throughout the course?**

This notebook is an original, standalone teaching chapter. It uses the textbook for structure and mathematical orientation, but the explanations, code, and figures are created here. The chapter focus is manifolds, Lie groups, Lie algebras, SE(3), adjoints, invariant volume. The key objects are manifolds, charts, tangent vectors, cotangent covectors, Lie groups, Lie algebras, exponential coordinates, adjoint actions, coadjoint actions, and invariant volume.

Generated artifacts live under `artifacts/appendix-a` and are displayed both by code and in the static gallery. The final sanity cell checks the mathematical claims and artifact files so that the notebook remains auditable after reruns.


## Translation Guide

The appendix supplies the geometry behind the course conventions. SO(3) and SE(3) are not just matrix sets; they are groups whose tangent spaces encode angular and rigid-body velocity. The computational translation used here is deliberately concrete: every named object becomes an array, graph, cone, trajectory, or map whose dimensions can be checked. That keeps the notebook independent from the PDF while preserving the chapter's mathematical route.

The main objects for this chapter are manifolds, charts, tangent vectors, cotangent covectors, Lie groups, Lie algebras, exponential coordinates, adjoint actions, coadjoint actions, and invariant volume. Read each one as a map between spaces. Ask what frame or chart supplies its coordinates, what operation changes those coordinates, and what quantity should remain invariant. The visuals in this notebook are chosen to make those questions inspectable rather than decorative.

The source pages are used only as orientation. Definitions and examples are paraphrased into course language, and all figures are generated from fresh code. When a visual resembles a textbook concept, it is a redraw or computational experiment rather than a page crop.


## Route Through The Chapter

We move in four passes. First, the notebook names the chapter's geometric objects and their engineering purpose. Second, it generates the visual sequence: manifold chart tangent view, se3 exponential screw reference, adjoint coadjoint power invariance. Third, it runs a compact computational check that records the relevant residuals, ranks, endpoint errors, determinants, or signs. Fourth, it closes with an applied lab that asks the reader to change a parameter and explain what stayed invariant.

The important habit is to compare the visual artifact with the JSON check. If a cone claims feasibility, the feasibility check should agree. If a Jacobian claims usable motion, the task-space determinant or rank should agree. If a loop claims bracket displacement, the endpoint check should reveal it. The notebook is designed so those cross-checks are near each other.


In [ ]:
from pathlib import Path
import sys
import numpy as np

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the robotics course root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
TOPIC = "appendix-a"

from utils.artifacts import display_artifact, save_json
from utils.visuals import build_storyboard, storyboard_check_payload


In [ ]:
storyboard = {
  "label": "Appendix A: Lie Groups and Robot Kinematics",
  "artifact_topic": "appendix-a",
  "source_span": "printed pp. 403-434; PDF pp. 421-452",
  "visual_sequence": [
    {
      "kind": "lie",
      "concept": "Manifold Chart Tangent View",
      "filename": "manifold-chart-tangent-view.png",
      "observation": "charts, tangent vectors, and covectors"
    },
    {
      "kind": "screw",
      "concept": "Se3 Exponential Screw Reference",
      "filename": "se3-exponential-screw-reference.png",
      "observation": "twist exponential and helical path"
    },
    {
      "kind": "adjoint",
      "concept": "Adjoint Coadjoint Power Invariance",
      "filename": "adjoint-coadjoint-power-invariance.png",
      "observation": "twist and wrench transformations"
    }
  ]
}

visual_results = build_storyboard(storyboard, ARTIFACT_ROOT, TOPIC)
visual_payload = storyboard_check_payload(storyboard, visual_results)
for item in visual_results:
    display_artifact(item["path"], width=720)
visual_payload


## Static Artifact Gallery

![Manifold Chart Tangent View](../../artifacts/appendix-a/figures/manifold-chart-tangent-view.png)

*Inspection target:* charts, tangent vectors, and covectors.

![Se3 Exponential Screw Reference](../../artifacts/appendix-a/figures/se3-exponential-screw-reference.png)

*Inspection target:* twist exponential and helical path.

![Adjoint Coadjoint Power Invariance](../../artifacts/appendix-a/figures/adjoint-coadjoint-power-invariance.png)

*Inspection target:* twist and wrench transformations.


## Worked Interpretation

The invariant-power computation repeats the Chapter 2 frame-change check, but now it is read as the adjoint/coadjoint pairing of a Lie group action. This is the chapter's small worked example, not a full simulator. It is intentionally narrow enough that the relevant convention can be seen directly in code and broad enough to catch the common failure mode.

The visual sequence and the executable check use compatible parameters whenever possible. The point is to avoid a split course where the images tell one story and the numbers tell another. When extending this notebook, reuse that pattern: pick one invariant, draw the geometry that exposes it, then save a check payload that would fail if the convention were reversed or the rank condition were lost.

**Pitfall to watch:** A Euclidean dot product on a six-vector is not automatically invariant under a translated frame. The appendix is where that warning becomes a structural fact rather than an isolated sign convention. This pitfall is why the notebook avoids silent coordinate conventions. Names, frames, dimensions, and signs are part of the teaching surface, not implementation trivia.


In [ ]:
from utils.lie import adjoint, coadjoint, instantaneous_power, se3_exp, se3_log, so3_exp

w = np.array([0.2, -0.1, 0.3])
R = so3_exp(w)
xi = np.array([0.15, -0.05, 0.22, 0.4, 0.1, -0.2])
T = se3_exp(xi)
V = np.array([0.3, -0.1, 0.25, 0.2, 0.4, -0.1])
F = np.array([0.5, 0.2, -0.3, 0.1, -0.4, 0.6])
Ad = adjoint(T)
F_new = np.linalg.inv(Ad).T @ F
check_payload = {
    "rotation_orthogonality_error": float(np.linalg.norm(R.T @ R - np.eye(3))),
    "rotation_det": float(np.linalg.det(R)),
    "se3_roundtrip_error": float(np.linalg.norm(se3_log(T) - xi)),
    "power_invariance_error": float(abs(instantaneous_power(F, V) - instantaneous_power(F_new, Ad @ V))),
    "coadjoint_shape": list(coadjoint(T).shape),
}
assert check_payload["rotation_orthogonality_error"] < 1e-10
assert abs(check_payload["rotation_det"] - 1.0) < 1e-10
assert check_payload["power_invariance_error"] < 1e-10
check_payload


## Applied Lab

Lab prompt: Translate a twist and wrench through a displaced frame and verify invariant power.

Before running the lab, make a prediction in three parts: which geometric object is being changed, which representation will show the change most clearly, and which scalar check should move in the same direction. After running it, compare the prediction with the saved JSON payload under `artifacts/appendix-a/labs`.

Use the pitfall above as a diagnostic. If the visual and scalar check disagree, inspect the frame convention, normalization, rank threshold, sign convention, or parameter range. The best result is not merely a green check; it is a short explanation of why the check protects the chapter's main idea.


In [ ]:
lab_notes = {
    "lab": 'Translate a twist and wrench through a displaced frame and verify invariant power.',
    "source_orientation": "printed pp. 403-434; PDF pp. 421-452",
    "artifact_topic": TOPIC,
    "reader_task": "Change one parameter, rerun the visual cell, and explain which invariant did or did not change.",
}
lab_path = save_json(lab_notes, TOPIC, "labs", "applied-lab.json", root=ARTIFACT_ROOT)
display_artifact(lab_path)


In [ ]:
from pathlib import Path

visual_paths = [Path(item["path"]) for item in visual_results]
assert visual_payload["assertions"]["has_multiple_visuals"]
assert visual_payload["assertions"]["all_visuals_nonblank"]
assert all(path.exists() and path.stat().st_size > 1000 for path in visual_paths)

final_sanity = {
    "visual_payload": visual_payload,
    "checks": check_payload,
    "visual_artifact_count": len(visual_paths),
    "visual_artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in visual_paths],
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json", root=ARTIFACT_ROOT)
display_artifact(sanity_path)
final_sanity


## Practice And Inspection Notes

Use this chapter as a small laboratory, not as a static summary. The source span is printed pp. 403-434 and PDF pp. 421-452, but the working material is the notebook itself: the generated artifacts, the executable check payload, and the applied lab. Start by reading the chapter question again: **What Lie-group facts explain the coordinate rules used throughout the course?** Then identify which part of the visual sequence gives the most direct answer. For this chapter the focus is manifolds, Lie groups, Lie algebras, SE(3), adjoints, invariant volume, so the useful inspection targets are not generic screenshots; they are the ranks, cones, motions, frames, phase loops, energy shapes, or dependency arrows that make that focus concrete.

The `manifold chart tangent view` artifact uses the `lie` visual family; inspect it for charts, tangent vectors, and covectors. The `se3 exponential screw reference` artifact uses the `screw` visual family; inspect it for twist exponential and helical path. The `adjoint coadjoint power invariance` artifact uses the `adjoint` visual family; inspect it for twist and wrench transformations.

After viewing the artifacts, rerun the computational check cell and read the keys in `check_payload` as a miniature rubric. Each key should protect one concept from a plausible robotics mistake. A determinant or rank protects a singularity claim. A residual protects an equation or closure condition. A monotonicity flag protects a scale-law interpretation. An endpoint error protects a steering construction. A power-invariance error protects a frame transformation. If a check is near zero, ask why zero is the right target; if a check is positive, ask what physical or geometric margin it measures.

Try three variations. First, make a small parameter change that should preserve the chapter's main invariant, such as a coordinate-frame change, a harmless redraw scale, or a sample density increase. Second, make a parameter change that should stress the model, such as a near-singular joint pose, lower friction coefficient, larger controller delay, smaller bracket loop, or weaker tendon tension. Third, make a convention change deliberately, such as reversing a normal or swapping a body/spatial interpretation, and predict which check should fail. These variations turn the notebook from a solved example into a diagnostic tool.

When writing your own notes, use this sentence pattern: "The artifact shows ..., the computation checks ..., and the invariant is ... ." That pattern is intentionally repetitive because it catches vague understanding. A reader who can fill in those three blanks for this chapter can usually transfer the idea to a different mechanism, contact model, or control task without reopening the textbook.


## Takeaways

- Lie groups package poses and tangent vectors in one coherent language.
- Adjoint actions explain body/spatial velocity and wrench transformations.
- Invariant quantities are precious because many convenient coordinates are not themselves invariant.
- Revisit the saved artifacts after changing parameters; the strongest learning usually comes from explaining why the invariant survived.
